Problem Statement
Your team has a working RAG chatbot prototype in a notebook, and now needs to expose it as a web API so other services can call it over HTTP, following the productionization approach of FastAPI + Pydantic + logging. Since this task should not depend on a real OpenAI/GROQ account or a live vector database, you will wire the same API structure around a deterministic mock "retrieval" function described below.

Dataset
A tiny in-memory FAQ knowledge base (the mock "documents" your retrieval function searches over):

FAQ_DOCS = [
    {"doc_id": "doc_1", "text": "Employees may work from home up to 2 days per week with manager approval."},
    {"doc_id": "doc_2", "text": "The probation period for new employees is 3 months from the date of joining."},
    {"doc_id": "doc_3", "text": "Subscriptions can be cancelled anytime from the account settings page; refunds are prorated."},
    {"doc_id": "doc_4", "text": "Office Wi-Fi credentials are issued by IT on the employee's first day."}
]
Tasks
Define a Pydantic request model ChatRequest with a required question: str field, and a Pydantic response model ChatResponse with fields answer: str, source_documents: list[str], and request_id: str.
Implement a mock retrieval-and-answer function ask_rag(question: str) -> dict that performs a simple case-insensitive keyword match of the question's words against each FAQ_DOCS entry's text, picks the best-matching document(s) (the one(s) with the highest number of overlapping keywords, at least one), and returns a dict with an answer string built from the matched document's text, a source_documents list of the matched doc_ids, and a generated request_id (e.g., a UUID string). This mock function replaces a real OpenAI/GROQ + vector-database call — no external API key or network call is required or permitted for this task.
Create a FastAPI app with: a GET / health-check endpoint returning a simple status message, and a POST /chat endpoint that accepts a ChatRequest body, calls ask_rag, and returns a ChatResponse.
Add basic logging (using Python's logging module, not print) that logs each incoming question and the resulting request_id.
Expected Output
A POST /chat request with body {"question": "How many days can I work from home?"} should return a JSON response whose source_documents includes "doc_1" and whose answer is built from that document's text.

In [1]:
%pip install fastapi


   ---------------------------------------- 0/2 [starlette]
   ---------------------------------------- 0/2 [starlette]
   ---------------------------------------- 0/2 [starlette]
   ---------------------------------------- 0/2 [starlette]
   ---------------------------------------- 0/2 [starlette]
   ---------------------------------------- 0/2 [starlette]
   ---------------------------------------- 0/2 [starlette]
   -------------------- ------------------- 1/2 [fastapi]
   -------------------- ------------------- 1/2 [fastapi]
   -------------------- ------------------- 1/2 [fastapi]
   -------------------- ------------------- 1/2 [fastapi]
   -------------------- ------------------- 1/2 [fastapi]
   -------------------- ------------------- 1/2 [fastapi]
   -------------------- ------------------- 1/2 [fastapi]
   -------------------- ------------------- 1/2 [fastapi]
   -------------------- ------------------- 1/2 [fastapi]
   -------------------- ------------------- 1/2 [fastapi]


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import logging
import re
from typing import List
from uuid import uuid4

from fastapi import FastAPI
from pydantic import BaseModel
from fastapi.testclient import TestClient

In [4]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("mock_rag_api")

FAQ_DOCS = [
    {"doc_id": "doc_1", "text": "Employees may work from home up to 2 days per week with manager approval."},
    {"doc_id": "doc_2", "text": "The probation period for new employees is 3 months from the date of joining."},
    {"doc_id": "doc_3", "text": "Subscriptions can be cancelled anytime from the account settings page; refunds are prorated."},
    {"doc_id": "doc_4", "text": "Office Wi-Fi credentials are issued by IT on the employee's first day."},
]


class ChatRequest(BaseModel):
    question: str


class ChatResponse(BaseModel):
    answer: str
    source_documents: List[str]
    request_id: str


WORD_PATTERN = re.compile(r"\b\w+\b")


def tokenize(text: str) -> set[str]:
    return set(WORD_PATTERN.findall(text.lower()))


def ask_rag(question: str) -> dict:
    question_tokens = tokenize(question)
    best_score = 0
    best_matches = []

    for doc in FAQ_DOCS:
        overlap = question_tokens & tokenize(doc["text"])
        score = len(overlap)

        if score > best_score:
            best_score = score
            best_matches = [doc]
        elif score == best_score and score > 0:
            best_matches.append(doc)

    request_id = str(uuid4())

    if not best_matches:
        return {
            "answer": "No relevant FAQ entry was found for your question.",
            "source_documents": [],
            "request_id": request_id,
        }

    answer = " ".join(doc["text"] for doc in best_matches)
    return {
        "answer": answer,
        "source_documents": [doc["doc_id"] for doc in best_matches],
        "request_id": request_id,
    }


app = FastAPI(title="Mock RAG Chat API")


@app.get("/")
def health_check():
    return {"status": "ok", "message": "Mock RAG API is running."}


@app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest):
    logger.info("Incoming question: %s", request.question)
    result = ask_rag(request.question)
    logger.info("Generated request_id: %s", result["request_id"])
    return ChatResponse(**result)


client = TestClient(app)
sample_response = client.post("/chat", json={"question": "How many days can I work from home?"}).json()
sample_response

2026-07-19 20:59:59,195 - INFO - Incoming question: How many days can I work from home?
2026-07-19 20:59:59,198 - INFO - Generated request_id: e52d439e-bf00-4c68-b4e4-fd767c15cbb7
2026-07-19 20:59:59,203 - INFO - HTTP Request: POST http://testserver/chat "HTTP/1.1 200 OK"


{'answer': 'Employees may work from home up to 2 days per week with manager approval.',
 'source_documents': ['doc_1'],
 'request_id': 'e52d439e-bf00-4c68-b4e4-fd767c15cbb7'}